# Time Series Analysis on Air Passengers Dataset
---
This notebook demonstrates **AR, MA, ARMA, ARIMA, and SARIMA** models on the famous Air Passengers dataset.
We will go step by step:
- Load & visualize data
- Check stationarity
- Apply AR, MA, ARMA
- Apply ARIMA & SARIMA
- Forecast future values


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Load AirPassengers dataset
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url, parse_dates=['Month'], index_col='Month')

df.head()


In [ ]:
plt.figure(figsize=(10,5))
plt.plot(df, label='Passengers')
plt.title('Air Passengers Dataset')
plt.xlabel('Year')
plt.ylabel('Number of Passengers')
plt.legend()
plt.show()


In [ ]:
# Augmented Dickey-Fuller Test
result = adfuller(df['Passengers'])
print('ADF Statistic:', result[0])
print('p-value:', result[1])

if result[1] > 0.05:
    print("Series is Non-Stationary")
else:
    print("Series is Stationary")

# Differencing to make stationary
df_diff = df['Passengers'].diff().dropna()

plt.figure(figsize=(10,5))
plt.plot(df_diff)
plt.title('Differenced Series')
plt.show()

# Check stationarity again
result_diff = adfuller(df_diff)
print('ADF Statistic (Diff):', result_diff[0])
print('p-value (Diff):', result_diff[1])


## AutoRegression (AR) Model

In [ ]:
train = df['Passengers'][:100]
test = df['Passengers'][100:]

ar_model = AutoReg(train, lags=12).fit()
ar_pred = ar_model.predict(start=len(train), end=len(train)+len(test)-1)

plt.figure(figsize=(10,5))
plt.plot(train, label='Train')
plt.plot(test, label='Test')
plt.plot(ar_pred, label='AR Predictions')
plt.legend()
plt.show()


## ARIMA Model

In [ ]:
arima_model = ARIMA(train, order=(2,1,2))
arima_res = arima_model.fit()
arima_pred = arima_res.predict(start=len(train), end=len(train)+len(test)-1, typ='levels')

plt.figure(figsize=(10,5))
plt.plot(train, label='Train')
plt.plot(test, label='Test')
plt.plot(arima_pred, label='ARIMA Predictions')
plt.legend()
plt.show()


## SARIMA Model

In [ ]:
sarima_model = SARIMAX(train, order=(1,1,1), seasonal_order=(1,1,1,12))
sarima_res = sarima_model.fit(disp=False)
sarima_pred = sarima_res.predict(start=len(train), end=len(train)+len(test)-1, typ='levels')

plt.figure(figsize=(10,5))
plt.plot(train, label='Train')
plt.plot(test, label='Test')
plt.plot(sarima_pred, label='SARIMA Predictions')
plt.legend()
plt.show()


## Future Forecasting with SARIMA

In [ ]:
future_forecast = sarima_res.predict(start=len(df), end=len(df)+12, typ='levels')
plt.figure(figsize=(10,5))
plt.plot(df['Passengers'], label='Original')
plt.plot(future_forecast, label='Future Forecast')
plt.legend()
plt.show()
